# Semana 10: Modelos Pre-entrenados: BERT y GPT – Fine-tuning
## Notebook Conceptual (NB1) – Explorando BERT y GPT

**Propósito:** Utilizar la librería Hugging Face para cargar modelos pre-entrenados y explorar sus capacidades.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Cargar el tokenizer de BERT y tokenizar oraciones.
- Inspeccionar los tensores de entrada (input_ids, attention_mask, token_type_ids).
- Pasar texto por BERT y obtener embeddings de tokens y el token [CLS].
- Generar texto con GPT-2 usando pipeline.
- Comparar representaciones contextuales de una palabra en diferentes oraciones.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Instalamos transformers si es necesario
try:
    import transformers
    import torch
    print("Transformers ya instalado.")
except ImportError:
    !pip install transformers
    !pip install torch

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModel, pipeline

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Cargando BERT y su Tokenizer

Cargamos el modelo `bert-base-uncased` y su tokenizador correspondiente.

In [ ]:
# Cargar tokenizer y modelo BERT
print("Cargando BERT tokenizer...")
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)

print("BERT cargado correctamente.")

---
## 2. Tokenización con BERT

### 2.1. Tokenizar una oración simple

In [ ]:
# Oración de ejemplo
sentence = "I love natural language processing with BERT"

# Tokenizar
tokens = bert_tokenizer.tokenize(sentence)
print(f"Oración original: {sentence}")
print(f"Tokens: {tokens}")

# Convertir a IDs
input_ids = bert_tokenizer.convert_tokens_to_ids(tokens)
print(f"Input IDs: {input_ids}")

### 2.2. Tokenización con tensores de entrada completos

BERT espera tres tensores: `input_ids`, `attention_mask` y `token_type_ids`.

In [ ]:
# Tokenizar con return_tensors='pt' para obtener tensores PyTorch
encoded = bert_tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)

print("Claves del diccionario:", encoded.keys())
print("\ninput_ids:")
print(encoded['input_ids'])
print("\nattention_mask:")
print(encoded['attention_mask'])
print("\ntoken_type_ids:")
print(encoded['token_type_ids'])

# Decodificar de vuelta a texto
decoded = bert_tokenizer.decode(encoded['input_ids'][0])
print(f"\nDecodificado: {decoded}")

### Explicación de los tensores:

- **input_ids**: IDs de los tokens según el vocabulario de BERT.
- **attention_mask**: 1 para tokens reales, 0 para padding (cuando hay truncamiento/padding).
- **token_type_ids**: Indica si el token pertenece a la primera oración (0) o segunda (1). Útil para tareas con pares de oraciones.

---
## 3. Pasando por el modelo BERT

Obtenemos las representaciones de cada token y el embedding del token [CLS].

In [ ]:
# Mover tensores al dispositivo
input_ids = encoded['input_ids'].to(device)
attention_mask = encoded['attention_mask'].to(device)

# Pasar por el modelo
with torch.no_grad():
    outputs = bert_model(input_ids, attention_mask=attention_mask)

# outputs.last_hidden_state contiene los embeddings de cada token
token_embeddings = outputs.last_hidden_state
print(f"Forma de token_embeddings: {token_embeddings.shape}")
print("(batch_size, seq_len, hidden_dim)")

# Extraer el embedding del token [CLS] (primer token)
cls_embedding = token_embeddings[:, 0, :]
print(f"\nForma del embedding [CLS]: {cls_embedding.shape}")
print(f"Primeras 10 dimensiones del [CLS]: {cls_embedding[0, :10].cpu().numpy()}")

### 3.1. Visualizar los embeddings de los tokens

In [ ]:
# Obtener los tokens decodificados
tokens = bert_tokenizer.convert_ids_to_tokens(encoded['input_ids'][0].cpu())
print("Tokens:", tokens)

# Tomar los primeros 10 tokens y sus embeddings
num_tokens = min(10, len(tokens))
token_embeds = token_embeddings[0, :num_tokens, :].cpu().numpy()

# Visualizar como heatmap
plt.figure(figsize=(14, 6))
sns.heatmap(token_embeds.T, cmap='viridis', cbar=True)
plt.xlabel('Posición del token')
plt.ylabel('Dimensión del embedding')
plt.title('Embeddings de tokens (primeras 10 posiciones)')
plt.yticks([])
plt.show()

---
## 4. Representaciones Contextuales de una Palabra

Comparamos cómo cambia el embedding de la misma palabra en diferentes contextos.

In [ ]:
# Definir dos oraciones con la palabra 'bank' en diferentes contextos
sentences = [
    "I went to the bank to withdraw money.",
    "The river bank was covered in grass."
]

# Tokenizar ambas oraciones
encoded_pair = bert_tokenizer(sentences, return_tensors='pt', padding=True, truncation=True).to(device)

# Obtener representaciones
with torch.no_grad():
    outputs_pair = bert_model(**encoded_pair)

token_embeddings_pair = outputs_pair.last_hidden_state

# Encontrar la posición de 'bank' en cada oración
tokens1 = bert_tokenizer.convert_ids_to_tokens(encoded_pair['input_ids'][0].cpu())
tokens2 = bert_tokenizer.convert_ids_to_tokens(encoded_pair['input_ids'][1].cpu())

print(f"Tokens oración 1: {tokens1}")
print(f"Tokens oración 2: {tokens2}")

# Buscar 'bank' (puede ser 'bank' como token completo o parte de subpalabra)
def find_word_embedding(tokens, embeddings, word):
    for i, token in enumerate(tokens):
        if word in token:
            return embeddings[0, i, :].cpu().numpy()
    return None

emb1 = find_word_embedding(tokens1, token_embeddings_pair, 'bank')
emb2 = find_word_embedding(tokens2, token_embeddings_pair, 'bank')

if emb1 is not None and emb2 is not None:
    # Calcular similitud de coseno
    sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    print(f"\nSimilitud de coseno entre las representaciones de 'bank': {sim:.4f}")
    
    # Visualizar diferencias
    plt.figure(figsize=(12, 4))
    plt.plot(emb1[:50], label='Contexto financiero', alpha=0.7)
    plt.plot(emb2[:50], label='Contexto geográfico', alpha=0.7)
    plt.xlabel('Dimensión')
    plt.ylabel('Valor')
    plt.title('Comparación de embeddings de "bank" en diferentes contextos (primeras 50 dimensiones)')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No se encontró la palabra 'bank' en los tokens.")

---
## 5. Generación de Texto con GPT-2

Usamos el pipeline de Hugging Face para generar texto con GPT-2.

In [ ]:
# Cargar pipeline de generación de texto
print("Cargando pipeline de generación de texto con GPT-2...")
generator = pipeline('text-generation', model='gpt2', device=0 if torch.cuda.is_available() else -1)

# Generar texto a partir de un prompt
prompt = "The future of artificial intelligence is"

# Generar 3 secuencias diferentes
outputs = generator(prompt, max_length=50, num_return_sequences=3, temperature=0.8)

print(f"Prompt: {prompt}")
print("\n=== GENERACIONES ===\n")
for i, output in enumerate(outputs):
    print(f"Generación {i+1}:\n{output['generated_text']}\n")

### 5.1. Experimentando con diferentes parámetros de generación

In [ ]:
# Probar diferentes temperaturas
temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    outputs = generator(prompt, max_length=40, num_return_sequences=1, temperature=temp)
    print(f"Temperatura {temp}: {outputs[0]['generated_text']}\n")

---
## 6. Comparación de Diferentes Modelos (Opcional)

Probamos otros modelos de Hugging Face para diferentes tareas.

In [ ]:
# Pipeline de análisis de sentimiento
sentiment_pipeline = pipeline('sentiment-analysis')

test_sentences = [
    "I love this movie, it's fantastic!",
    "This film is terrible and boring."
]

print("=== ANÁLISIS DE SENTIMIENTO ===")
for sent in test_sentences:
    result = sentiment_pipeline(sent)
    print(f"Texto: {sent}")
    print(f"Resultado: {result}\n")

---
## 7. Ejercicio para el Estudiante

1. **Explora otras oraciones**: Prueba con diferentes oraciones y observa cómo cambian los embeddings de palabras polisémicas.

2. **Compara modelos**: Carga otro modelo de BERT (por ejemplo, 'bert-large-uncased') y compara el tamaño de los embeddings.

3. **Generación creativa**: Usa GPT-2 para generar poemas, historias cortas o código.

4. **Tokenización inversa**: Toma los input_ids de una oración, modifícalos manualmente y decodifica para ver el resultado.

5. **Visualización con PCA**: Usa PCA para reducir los embeddings de los tokens a 2D y visualiza cómo se agrupan palabras similares.

In [ ]:
# Espacio para el estudiante
# ...

---
## 8. Conclusiones

En este notebook hemos explorado las capacidades de los modelos pre-entrenados:

✔️ **BERT**:
- Aprendimos a tokenizar con WordPiece.
- Inspeccionamos los tensores de entrada (input_ids, attention_mask, token_type_ids).
- Obtuvimos embeddings de tokens y el vector [CLS].
- Comparamos representaciones contextuales de la misma palabra en diferentes contextos.

✔️ **GPT-2**:
- Usamos el pipeline de generación de texto.
- Experimentamos con parámetros como temperatura.

✔️ **Conceptos clave**:
- Los embeddings son contextuales, cambiando según el significado.
- La librería Hugging Face facilita el uso de modelos avanzados.

**Lección clave**: Los modelos pre-entrenados como BERT y GPT capturan matices del lenguaje que los embeddings estáticos no pueden, y Hugging Face nos permite usarlos fácilmente.

En el próximo notebook (NB2) realizaremos fine-tuning de BERT para una tarea específica.

---
**Fin del Notebook Conceptual - Semana 10 (NLP)**